In [ ]:
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()


In [ ]:
import jax
print('✅ JAX version:', jax.devices())
import jax
print('✅ JAX version:', jax.__version__)
print('✅ Devices:', jax.devices())
if any(d.platform == 'gpu' for d in jax.devices()):
    print('🎉 GPU is working!')
else:
    print('⚠️ No GPU detected')

In [ ]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=6 n=3.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_3 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_3.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_3.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_3.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_3.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=3, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

In [ ]:
import pennylane as qml
import jax
import jax.numpy as jnp
import numpy as np  # 用于预处理，非计算图部分
import time
import optax  # JAX 的标准优化库 (需要 pip install optax)
from catalyst import qjit, grad

# =================== 0. 辅助函数 (保持不变) ===================
def get_Hami(H):
    # 安全地将 H 转换为无梯度的 NumPy 数组
    if hasattr(H, 'toarray'):
        H = H.toarray()
    if hasattr(H, 'detach'):
        H = H.detach().cpu().numpy()
    elif hasattr(H, 'numpy'):
        H = H.numpy()
    else:
        H = np.asarray(H)

    H_dense = np.array(H, copy=True)
    d = H_dense.shape[0]
    Nq = int(np.ceil(np.log2(d)))
    l = 2 ** Nq

    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 1000)
    Hami[:d, :d] = H_dense

    return Hami, Nq

H_sy,Nq = get_Hami(H)

from scipy.linalg import eigh

# 假设 H_sy 是一个 n x n 的实对称矩阵
eigenvals, eigenvecs = eigh(H_sy)

# 输出所有特征值（已按升序排列）
print("所有特征值（对角化结果）:")
print(eigenvals)

min =  eigh( H_sy,eigvals_only=True)[0]
print(min)
print("最小特征值:", min_eigval[0])

In [ ]:
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np

H_sy,Nq = get_Hami(H)
H_sy = coo_matrix(H_sy)
# ---------- 1. 生成 Gray 码 ----------
def gray_code(n: int) -> list[str]:
    """返回 n-bit Gray 码列表，顺序对应 0..2^n-1"""
    if n == 1:
        return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis   = gray_code(Nq)                        # len == 2**Nq
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)

# ---------- 2. 构造 Gray-ordered 哈密顿量 ----------

rows, cols = H_sy.nonzero()          # 默认是 int32/uint32
data       = H_sy.data

# 关键修复：先把索引数组变成 int64，再做高级索引
rows = rows.astype(np.int64, copy=False)
cols = cols.astype(np.int64, copy=False)

new_rows = gray2natural[rows]
new_cols = gray2natural[cols]

print('data  len:', len(data))
print('rows  len:', len(new_rows))
print('cols  len:', len(new_cols))

# 现在三者长度一致，且索引不会溢出
H_gray = coo_matrix((data, (new_rows, new_cols)),
                    shape=(2**Nq, 2**Nq)).tocsr()

H_gray = H_gray.toarray()

# ---------- 3. 后续计算 ----------
from scipy.sparse.linalg import eigsh
eigenvals, eigenvecs = eigh(H_gray)

# 输出所有特征值（已按升序排列）
print("所有特征值（对角化结果）:")
print(eigenvals)





